# Baseline performance of Pubmedbert for exaggeration detection

In [8]:
import sys
from pyprojroot import here

sys.path.insert(0, str(here()))

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
import numpy as np

/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Pubmedbert model and tokenizer

In [2]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Create tokenize fold helper function

In [3]:
def tokenize_fold(df, max_length=256):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

### Create helper to calculate metrics

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

### Create model

In [5]:
def make_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
    )

### Set training arguments using standard default values, not tuning yet

In [6]:
training_args = TrainingArguments(
    output_dir="outputs/pubmedbert_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    warmup_ratio=0.1,
    fp16=True,
    seed=42,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### Train on 1 fold as a test

In [10]:
from src.data_holdout import load_hf_dataset, process_and_pool_data, get_fold_from_disk

full_data = load_hf_dataset("copenlu/scientific-exaggeration-detection")
full_df = process_and_pool_data(full_data["train"], full_data["test"])
train_fold, val_fold = get_fold_from_disk(full_df, fold=0, k=5, seed=7)

train_ds = tokenize_fold(train_fold, max_length=256)
val_ds = tokenize_fold(val_fold, max_length=256)

model = make_model()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3023.20it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | 

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.947295,0.250980
2,No log,0.964403,0.276188
3,No log,0.950915,0.370602
4,No log,0.978277,0.441480
5,No log,0.989764,0.457275


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.21it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

{'eval_loss': 0.9897643327713013, 'eval_macro_f1': 0.4572751322751323, 'eval_runtime': 9.3371, 'eval_samples_per_second': 11.353, 'eval_steps_per_second': 1.499, 'epoch': 5.0}


### Run on all 5 folds

In [11]:
train_folds, val_folds = get_fold_from_disk(full_df, k=5, seed=7)

fold_scores = []

for fold in range(5):
    print(f"\n===== Fold {fold} =====")

    train_ds = tokenize_fold(train_folds[fold], max_length=256)
    val_ds = tokenize_fold(val_folds[fold], max_length=256)

    model = make_model()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    metrics = trainer.evaluate()

    fold_f1 = metrics["eval_macro_f1"]
    fold_scores.append(fold_f1)

    print(f"Fold {fold} macro_f1 = {fold_f1:.4f}")

print("\nPubMedBERT baseline CV result:")
print(f"{np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")


===== Fold 0 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1547.58it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | 

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.927252,0.253411
2,No log,0.897949,0.276822
3,No log,0.919365,0.436335
4,No log,0.983986,0.413909
5,No log,0.999841,0.418515


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.07it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

Fold 0 macro_f1 = 0.4363

===== Fold 1 =====


...

Fold 3 macro_f1 = 0.3809

===== Fold 4 =====


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2982.47it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | 

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.967169,0.272196
2,No log,0.909858,0.253411
3,No log,0.895499,0.316218
4,No log,0.895797,0.403680
5,No log,0.899162,0.422157


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]
/home/rebecca_m/miniforge3/envs/mids266/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1

Fold 4 macro_f1 = 0.4222

PubMedBERT baseline CV result:
0.4089 ± 0.0469
